# Fine-Tuning mT5-small for Turkish Summarization

This notebook fine-tunes `google/mt5-small` on the Turkish subset of XLSum dataset.
After training, the model will be much better at summarizing Turkish text without hallucinating.

**Requirements:** T4 GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets sentencepiece protobuf accelerate rouge-score

In [ ]:
# Step 2: Verify GPU is available
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Step 3: Load the Turkish subset of XLSum dataset
from datasets import load_dataset

# XLSum contains news articles with summaries in 45 languages
dataset = load_dataset('csebuetnlp/xlsum', 'turkish')
print(f'Train: {len(dataset["train"])} examples')
print(f'Validation: {len(dataset["validation"])} examples')
print(f'Test: {len(dataset["test"])} examples')
print()
print('Example:')
print(f'Title: {dataset["train"][0]["title"]}')
print(f'Text: {dataset["train"][0]["text"][:200]}...')
print(f'Summary: {dataset["train"][0]["summary"]}')

In [ ]:
# Step 4: Load model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print(f'Model loaded: {model_name}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# Step 5: Preprocess the dataset
import re

WHITESPACE_HANDLER = lambda k: re.sub(r'\s+', ' ', re.sub(r'\n+', ' ', k.strip()))

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    inputs = [WHITESPACE_HANDLER(text) for text in examples['text']]
    targets = [WHITESPACE_HANDLER(summary) for summary in examples['summary']]

    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding='max_length'
    )
    labels = tokenizer(
        targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding='max_length'
    )
    # Replace pad token id with -100 so it's ignored in loss
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Use a subset for faster training (full dataset takes hours)
train_subset = dataset['train'].select(range(5000))
val_subset = dataset['validation'].select(range(500))

tokenized_train = train_subset.map(preprocess_function, batched=True, remove_columns=train_subset.column_names)
tokenized_val = val_subset.map(preprocess_function, batched=True, remove_columns=val_subset.column_names)

print(f'Tokenized train: {len(tokenized_train)}')
print(f'Tokenized val: {len(tokenized_val)}')

In [ ]:
# Step 6: Set up training
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir='./mt5-turkish-summarization',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=200,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print('Training setup complete. Starting training...')

In [ ]:
# Step 7: TRAIN (this takes ~30-45 minutes on T4 GPU)
trainer.train()
print('Training complete!')

In [ ]:
# Step 8: Test the fine-tuned model
from transformers import pipeline

summarizer = pipeline('summarization', model=model, tokenizer=tokenizer, device=0)

test_text = (
    "Yapay zeka, makinelerin insan benzeri zeka gösterme yeteneğidir. "
    "Günümüzde yapay zeka, sağlık, eğitim, ulaşım ve iletişim gibi birçok alanda "
    "hayatımızı değiştirmektedir. Makine öğrenmesi, derin öğrenme ve doğal dil işleme "
    "gibi alt dallar, yapay zekanın temel bileşenleridir. Sağlık alanında yapay zeka, "
    "hastalıkların erken teşhisinde ve kişiselleştirilmiş tedavi planlamada kullanılmaktadır."
)

result = summarizer(WHITESPACE_HANDLER(test_text), max_length=80, min_length=20)
print('Test Summary:')
print(result[0]['summary_text'])

In [ ]:
# Step 9: Save the fine-tuned model
save_path = './mt5-turkish-finetuned'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f'Model saved to {save_path}')

In [ ]:
# Step 10: Zip and download the model
!zip -r mt5-turkish-finetuned.zip ./mt5-turkish-finetuned

from google.colab import files
files.download('mt5-turkish-finetuned.zip')
print('Download started! Save it to your diploma_project/models/ folder')

## After downloading:

1. Unzip `mt5-turkish-finetuned.zip` into your project: `diploma_project/models/mt5-turkish-finetuned/`
2. The app will automatically detect and use it for Turkish text

## What this fine-tuning does:
- Trains mT5-small on 5000 Turkish news articles + their human-written summaries
- The model learns what good Turkish summaries look like
- After fine-tuning: less hallucination, better grammar, more relevant content
- Training takes ~30-45 minutes on T4 GPU